# 🚀 CIFAR-10 - Best Practices 2025

## State-of-the-Art Implementation

**Ziel:** 90%+ Test Accuracy durch moderne Techniken

**Verbesserungen gegenüber Original:**
1. ✅ SGD + Cosine LR Schedule (statt Adam)
2. ✅ Augmented Data Pipeline
3. ✅ Wide ResNet-28-10 (~36M params)
4. ✅ Weight Decay + Label Smoothing
5. ✅ 200 Epochen Training

**Performance-Vergleich:**
- Original Custom CNN: 79.75%
- Erwartung hier: **90-94%**

---

**Quellen:**
- Wide ResNet (BMVC 2016)
- Papers with Code CIFAR-10
- Cosine Annealing (SGDR 2016)

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow: {tf.__version__}")

# GPU Check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ {len(gpus)} GPU(s) verfügbar")
else:
    print("⚠️ Keine GPU - läuft auf CPU")

# Mixed Precision
keras.mixed_precision.set_global_policy('mixed_float16')
print("✅ Mixed Precision aktiviert")

# Seeds
np.random.seed(42)
tf.random.set_seed(42)

print("Setup abgeschlossen! 🚀")

## 2. Daten laden

In [ ]:
# CIFAR-10 laden
(train_images, train_labels), (test_images, test_labels) = cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Train: {train_images.shape}")
print(f"Test: {test_images.shape}")

# Normalisieren
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0

# Labels
train_labels_flat = train_labels.flatten()
test_labels_flat = test_labels.flatten()
train_labels_onehot = to_categorical(train_labels_flat, 10)
test_labels_onehot = to_categorical(test_labels_flat, 10)

print("✅ Daten normalisiert")

## 3. Quick EDA

In [ ]:
# Sample Images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(train_images[i])
    ax.set_title(class_names[train_labels_flat[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. 🎨 Data Augmentation

**Moderne Augmentation:**
- RandomFlip (horizontal)
- RandomRotation (±15°)
- RandomZoom (±15%)
- RandomTranslation (±15%)
- RandomContrast (±20%)

**GPU-optimiert:** Keras Layers statt ImageDataGenerator

In [ ]:
# Sample Images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    # Fix: Explizite Konvertierung zu float32, da float16 von Matplotlib nicht unterstützt wird
    plot_image = train_images[i].astype(np.float32)

    ax.imshow(plot_image)
    ax.set_title(class_names[train_labels_flat[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Data Augmentation Pipelinedata_augmentation = keras.Sequential([    layers.RandomFlip("horizontal"),    layers.RandomRotation(0.15),    layers.RandomZoom(0.15),    layers.RandomTranslation(0.15, 0.15),    layers.RandomContrast(0.2),], name='augmentation')print("✅ Augmentation Pipeline erstellt")# Visualizesample = train_images[0:1]fig, axes = plt.subplots(2, 5, figsize=(15, 6))fig.suptitle('Augmentation Examples')for ax in axes.flat:    aug = data_augmentation(sample, training=True)[0]    # Convert to float32 for matplotlib    aug_float32 = tf.cast(aug, tf.float32).numpy()    ax.imshow(aug_float32)    ax.axis('off')plt.tight_layout()plt.show()

## 5. 🏗️ Wide ResNet-28-10

**Architektur:**
- 28 Layer tief
- 10× breiter als Standard ResNet
- ~36M Parameter
- State-of-the-Art: ~96% auf CIFAR-10

**Residual Blocks:**
```
Input → BN → ReLU → Conv3x3 → BN → ReLU → Dropout → Conv3x3 → Add(Shortcut)
```

In [ ]:
# Muster für Data Augmentation Pipeline
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomRotation(factor=0.07),
    layers.RandomZoom(height_factor=0.1, width_factor=0.1),
    layers.RandomContrast(factor=0.1)
], name="data_augmentation")

print("✅ Data Augmentation Pipeline created (used in model)")

In [ ]:
def wide_residual_block(x, filters, strides=1, dropout_rate=0.3):
    """Wide Residual Block"""
    shortcut = x
    
    # Main path
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, strides=strides, padding='same', use_bias=False)(x)
    
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    
    # Adjust shortcut
    if strides != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=strides, use_bias=False)(shortcut)
    
    # Residual connection
    x = layers.Add()([x, shortcut])
    return x

def build_wide_resnet(depth=28, width_multiplier=10, num_classes=10, dropout_rate=0.3):
    """Build Wide ResNet-28-10"""
    assert (depth - 4) % 6 == 0
    n = (depth - 4) // 6
    
    filters = [16, 16 * width_multiplier, 32 * width_multiplier, 64 * width_multiplier]
    
    inputs = layers.Input(shape=(32, 32, 3), dtype='float32')
    x = data_augmentation(inputs)
    x = layers.Conv2D(filters[0], 3, padding='same', use_bias=False)(x)
    
    # Group 1: 32x32
    for i in range(n):
        x = wide_residual_block(x, filters[1], dropout_rate=dropout_rate)
    
    # Group 2: 16x16
    for i in range(n):
        strides = 2 if i == 0 else 1
        x = wide_residual_block(x, filters[2], strides=strides, dropout_rate=dropout_rate)
    
    # Group 3: 8x8
    for i in range(n):
        strides = 2 if i == 0 else 1
        x = wide_residual_block(x, filters[3], strides=strides, dropout_rate=dropout_rate)
    
    # Output
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    
    return models.Model(inputs, outputs, name=f'WideResNet-{depth}-{width_multiplier}')

# Build Model
model = build_wide_resnet(depth=28, width_multiplier=10)
print(f"✅ Wide ResNet-28-10 created")
print(f"   Parameters: {model.count_params():,}")

## 6. ⚙️ Compilation - Best Practices 2025

**Optimizer:** SGD + Nesterov Momentum
- Learning Rate: 0.1 → 0.0 (Cosine Decay)
- Momentum: 0.9
- Nesterov: True
- Weight Decay: 5e-4 (L2 Regularization)

**Loss:** Categorical Crossentropy + Label Smoothing (0.1)

**LR Schedule:** Cosine Annealing (smooth decay über 200 Epochen)

In [ ]:
# Hyperparameters
EPOCHS = 200
BATCH_SIZE = 128
INITIAL_LR = 0.1
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.1

steps_per_epoch = len(train_images) // BATCH_SIZE
total_steps = EPOCHS * steps_per_epoch

# Cosine LR Schedule
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=INITIAL_LR,
    decay_steps=total_steps,
    alpha=0.0
)

# SGD Optimizer
optimizer = keras.optimizers.SGD(
    learning_rate=lr_schedule,
    momentum=0.9,
    nesterov=True
)

# Weight Decay (L2 Regularization)
for layer in model.layers:
    if isinstance(layer, (layers.Conv2D, layers.Dense)):
        layer.kernel_regularizer = keras.regularizers.l2(WEIGHT_DECAY)

# Loss with Label Smoothing
loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

# Compile
model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

print("✅ Model kompiliert!")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Initial LR: {INITIAL_LR}")
print(f"Optimizer: SGD (momentum=0.9, nesterov=True)")
print(f"LR Schedule: Cosine Decay")
print(f"Weight Decay: {WEIGHT_DECAY}")
print(f"Label Smoothing: {LABEL_SMOOTHING}")

## 7. 🚀 Training

In [ ]:
import time
import tensorflow as tf
from tensorflow import keras
import numpy as np

# --- 1. SETUP UND DATEN-PIPELINE ---
# Bitte passen Sie diese Variablen an Ihre tatsächlichen Werte an!
EPOCHS = 200
BATCH_SIZE = 128

# --- PRÜFUNG DER VORAUSSETZUNGEN ---
# Wird nur zur Laufzeitprüfung verwendet, da ich keinen Zugriff auf die geladenen Daten habe.
if 'train_images' not in locals():
    print("🚨 HINWEIS: 'train_images' wurde nicht gefunden. Mit Platzhalter-Daten fortfahren.")
    train_images = np.zeros((50000, 32, 32, 3), dtype=np.float32)
    train_labels_onehot = np.zeros((50000, 10), dtype=np.float32)

# Daten in ein Dataset umwandeln
full_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels_onehot))

# Manuelle Aufteilung in Trainings- und Validierungsdaten (20% Validierung)
DATASET_SIZE = len(train_images)
VAL_SIZE = int(DATASET_SIZE * 0.2)

# Shuffeln des gesamten Datensatzes
full_dataset = full_dataset.shuffle(buffer_size=1024, reshuffle_each_iteration=True)

# Datasets erstellen und Prefetching für optimale Leistung anwenden
val_dataset = full_dataset.take(VAL_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train_dataset = full_dataset.skip(VAL_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

total_steps_per_epoch = tf.data.experimental.cardinality(train_dataset).numpy()
print(f"✅ Optimierte tf.data.Datasets erstellt.")

# --- 2. DYNAMISCHE ZEITERPROBUNG UND KALKULATION ---
print("\n" + "=" * 60)
print("⏱️ Starte dynamische Zeiterprobung über die ersten Batches...")
print("=" * 60)

TRIAL_STEPS = 10
total_trial_time = 0

if 'model' not in locals() or not hasattr(model, 'train_step'):
    print("🚨 FATALER FEHLER: Das 'model' ist nicht kompiliert oder nicht definiert.")
    avg_sec_per_step = 0
    total_estimated_time_min = 0.0
else:
    for step, (x_batch_train, y_batch_train) in enumerate(train_dataset.take(TRIAL_STEPS)):
        start_step_time = time.time()

        # KORRIGIERTE SYNTAX: Übergabe des Batches als EINZIGES Tupel (x, y)
        model.train_step((x_batch_train, y_batch_train))

        total_trial_time += (time.time() - start_step_time)

        if step == TRIAL_STEPS - 1:
            break

    if total_trial_time > 0:
        avg_sec_per_step = total_trial_time / TRIAL_STEPS
        estimated_sec_per_epoch = avg_sec_per_step * total_steps_per_epoch
        total_estimated_time_min = (estimated_sec_per_epoch * EPOCHS) / 60

        print(f"\n✅ Erprobung abgeschlossen. Durchschnittliche Zeit pro Schritt: {avg_sec_per_step:.4f} Sekunden")
        print("=========================================================================")
        print(f"➡️ Geschätzte Gesamtdauer für {EPOCHS} Epochen: {total_estimated_time_min:.1f} Minuten")
        print("=========================================================================")
    else:
        avg_sec_per_step = 0
        total_estimated_time_min = 0.0


# --- 3. BESCHLEUNIGTES TRAINING ---
if total_estimated_time_min > 0:

    # Callbacks
    checkpoint = keras.callbacks.ModelCheckpoint(
        'best_wide_resnet.keras',
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    )

    # histogram_freq=0 reduziert den Overhead
    tensorboard = keras.callbacks.TensorBoard(log_dir='./logs', histogram_freq=0)

    print(f"\nTraining Wide ResNet-28-10 for {EPOCHS} epochs...")
    print(f"Batch Size: {BATCH_SIZE}")
    print(f"➡️ Geschätzte Gesamtdauer: {total_estimated_time_min:.1f} Minuten")
    print("-" * 60)

    start_time = time.time()

    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=val_dataset,
        callbacks=[checkpoint, tensorboard],
        verbose=1 # Behält die gewünschte tqdm-Progress-Bar bei
    )

    training_time = time.time() - start_time

    print(f"\n✅ Training complete!")
    print("=========================================================================")
    print(f"   Tatsächliche Gesamtdauer: {training_time/60:.1f} Minuten")
    print(f"   Durchschnittliche Zeit pro Epoche: {training_time/EPOCHS:.1f} Sekunden")
    print("=========================================================================")

In [ ]:
import time

# Callbacks
checkpoint = keras.callbacks.ModelCheckpoint(
    'best_wide_resnet.keras',
    save_best_only=True,
    monitor='val_accuracy',
    verbose=1
)

tensorboard = keras.callbacks.TensorBoard(log_dir='./logs', histogram_freq=1)

# Training
print(f"Training Wide ResNet-28-10 for {EPOCHS} epochs...")
print(f"Batch Size: {BATCH_SIZE}, Steps per Epoch: {steps_per_epoch}")

start_time = time.time()

history = model.fit(
    train_images, train_labels_onehot,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=[checkpoint, tensorboard],
    verbose=1
)

training_time = time.time() - start_time

print(f"✅ Training complete!")
print(f"   Total Time: {training_time/60:.1f} min")
print(f"   Time per Epoch: {training_time/EPOCHS:.1f} sec")

## 8. 📊 Evaluation

In [ ]:
# Load best model
model = keras.models.load_model('best_wide_resnet.keras')

# Evaluate
print("Evaluating on test set...")
test_loss, test_acc = model.evaluate(test_images, test_labels_onehot, verbose=0)

print("=" * 60)
print("🏆 FINAL RESULTS")
print("=" * 60)
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")
print("=" * 60)

# Predictions
predictions = model.predict(test_images, verbose=0)
predicted_classes = np.argmax(predictions, axis=1)

# Confusion Matrix
cm = confusion_matrix(test_labels_flat, predicted_classes)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix - Accuracy: {test_acc*100:.2f}%', fontsize=16, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Classification Report
print("" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(test_labels_flat, predicted_classes, target_names=class_names))

## 9. 📈 Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val', linewidth=2)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val', linewidth=2)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Best Epoch
best_epoch = np.argmax(history.history['val_accuracy'])
best_val_acc = max(history.history['val_accuracy'])
print(f"Best Validation Accuracy: {best_val_acc*100:.2f}% (Epoch {best_epoch+1})")

## 10. 📋 Summary

In [ ]:
print("=" * 80)
print("🎯 CIFAR-10 BEST PRACTICES 2025 - FINAL SUMMARY")
print("=" * 80)
print(f"Architecture: Wide ResNet-28-10")
print(f"Parameters: {model.count_params():,}")
print(f"Training Configuration:")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Training Time: {training_time/60:.1f} min")
print(f"Optimization:")
print(f"  - Optimizer: SGD (momentum=0.9, nesterov=True)")
print(f"  - Initial LR: {INITIAL_LR}")
print(f"  - LR Schedule: Cosine Decay")
print(f"  - Weight Decay: {WEIGHT_DECAY}")
print(f"  - Label Smoothing: {LABEL_SMOOTHING}")
print(f"Regularization:")
print(f"  - Data Augmentation: ✅")
print(f"  - Dropout: 0.3")
print(f"  - Weight Decay: {WEIGHT_DECAY}")
print(f"  - Label Smoothing: {LABEL_SMOOTHING}")
print(f"🏆 RESULTS:")
print(f"  - Test Accuracy: {test_acc * 100:.2f}%")
print(f"  - Test Loss: {test_loss:.4f}")
print(f"  - Best Val Accuracy: {best_val_acc*100:.2f}% (Epoch {best_epoch+1})")
print("=" * 80)
print(f"✅ Target erreicht: {'JA!' if test_acc >= 0.90 else 'Nein (Ziel: 90%+)'}